# Carrier-Phase GNSS: RTK and PPP-RTK

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/RtkAndPppExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
Carrier-phase GNSS reaches mm-cm accuracy, but only after resolving the integer **ambiguity** of each phase measurement. This notebook shows the two classic flavours, both built from GTSAM factors and resolved with LAMBDA:

- **Part 1 - RTK** (double difference): a nearby base station differences away the satellite/receiver clocks and the atmosphere, leaving position + integer ambiguities. Uses `DoubleDifferencePseudorangeFactor` / `DoubleDifferenceCarrierPhaseFactor`.
- **Part 2 - PPP-RTK** (undifferenced): no base station; the receiver clock, tropospheric ZTD and slant ionosphere are carried as states and corrected with QZSS CLAS. Uses `UndifferencedPseudorangeFactor` / `UndifferencedCarrierPhaseFactor`.

The GNSS front-end (RINEX / SSR decoding, satellite states, geometry) is provided by [cssrlib](https://github.com/inuex35/cssrlib-numba); GTSAM builds the factor graph (incremental ISAM2) and the integers are resolved with `resamb_lambda`. Part 1 runs on data bundled with cssrlib; Part 2 downloads a short open-sky CLAS record.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

In [1]:
# Colab: install GTSAM (with the PPP factors) and the cssrlib GNSS front-end,
# then download the open-sky QZSS CLAS dataset. The undifferenced factors are
# new, so until this PR is merged Colab needs GTSAM built from this branch.
import os
import urllib.request

try:
    import google.colab  # noqa: F401
    !pip install --quiet numpy gtsam-develop \
        "git+https://github.com/inuex35/cssrlib-numba.git@claude/gtsam-ppprtk-minimal"
except ImportError:
    pass

# Set CSSRLIB_DATA to reuse a local copy; otherwise the files are downloaded here.
DATADIR = os.environ.get("CSSRLIB_DATA", "gnss_data")
_BASE = "https://raw.githubusercontent.com/hirokawa/cssrlib-data/main/data"
_FILES = {
    "doy2025-233/233h_rnx.obs":   f"{_BASE}/doy2025-233/233h_rnx.obs",
    "doy2025-233/233h_rnx.nav":   f"{_BASE}/doy2025-233/233h_rnx.nav",
    "doy2025-233/233h_qzsl6.txt": f"{_BASE}/doy2025-233/233h_qzsl6.txt",
    "clas_grid.def":              f"{_BASE}/clas_grid.def",
    "antex/igs20.atx":            "https://files.igs.org/pub/station/general/igs20.atx",
}
for rel, url in _FILES.items():
    dst = os.path.join(DATADIR, rel)
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        print("downloading", rel, "...")
        urllib.request.urlretrieve(url, dst)
print("data ready in", DATADIR)

data ready in /home/ubuntu/cssrlib/cssrlib-data/data


## Part 1 - RTK (double difference, base + rover)

Two receivers (a static rover and a base at a surveyed location) observe the same satellites. Differencing between receivers and between satellites cancels the satellite and receiver clocks and most of the atmosphere, so only the static rover position and the between-receiver integer ambiguities remain. With a short baseline this fixes essentially instantaneously. Runs on the dataset bundled with the cssrlib package (no download).

In [2]:
import os
import numpy as np

import cssrlib.rinex as rn
import cssrlib.gnss as gn
from cssrlib.gnss import rSigRnx, uTYP, sat2prn
from cssrlib.rtk import rtkpos
import gtsam
from gtsam import symbol

SYSS = (gn.uGNSS.GPS, gn.uGNSS.GAL)        # constellations to use
X = symbol('x', 0)                         # static rover ECEF position node
def AM(sat, f): return symbol('n', int(sat) * 10 + f)  # SD ambiguity node

### 1. Load RINEX (rover, base, navigation)

`rtkpos` is the cssrlib double-difference engine. We seed `nav.x[0:3]` with the
approximate rover position so its `qcedit` can compute satellite elevations.
`xyz_ref` is the surveyed marker used only to score accuracy.

In [3]:
bdir = os.path.join(os.path.dirname(gn.__file__), 'data') + os.sep  # bundled with cssrlib
xyz_ref = np.array([-3962108.673, 3381309.574, 3668678.638])
pos_ref = gn.ecef2pos(xyz_ref)

sigs = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("GL1C"), rSigRnx("GL2W"),
        rSigRnx("GS1C"), rSigRnx("GS2W"),
        rSigRnx("EC1C"), rSigRnx("EC5Q"), rSigRnx("EL1C"), rSigRnx("EL5Q"),
        rSigRnx("ES1C"), rSigRnx("ES5Q")]
sigsb = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("GL1C"), rSigRnx("GL2W"),
         rSigRnx("GS1C"), rSigRnx("GS2W"),
         rSigRnx("EC1X"), rSigRnx("EC5X"), rSigRnx("EL1X"), rSigRnx("EL5X"),
         rSigRnx("ES1X"), rSigRnx("ES5X")]

dec = rn.rnxdec(); dec.setSignals(sigs)
nav = gn.Nav(); dec.decode_nav(bdir + 'SEPT078M.21P', nav)
decb = rn.rnxdec(); decb.setSignals(sigsb)
decb.decode_obsh(bdir + '3034078M1.21O'); dec.decode_obsh(bdir + 'SEPT078M1.21O')
nav.rb = [-3959400.631, 3385704.533, 3667523.111]    # base station ECEF
rb = np.array(nav.rb)
rtk = rtkpos(nav, dec.pos)
nav.x[0:3] = np.array(dec.pos)                        # seed for elevations
nf = nav.nf
print('frequencies per constellation:', nf, ' base-rover baseline:',
      round(np.linalg.norm(xyz_ref - rb) / 1e3, 2), 'km')

frequencies per constellation: 2  base-rover baseline: 5.29 km


### 2. Front-end: per-epoch double-difference measurements

`prepare_double_difference_measurements` returns, per epoch, the rover/base
satellite positions (`rs`/`rsb`), the common-satellite indices (`iu`/`ir`),
the common satellites (`sat`) and rover elevations (`el`). No EKF -- this is just
the geometry/observation bundle that the GTSAM factors consume.

In [4]:
frames = []
sync = rn.sync_obs_hold(dec, decb, maxage=nav.maxtdiff)
for ne, (obs, obsb, dt) in enumerate(sync):
    if ne >= 60:
        break
    if obsb is None:
        continue
    dd = rtk.prepare_double_difference_measurements(obs, obsb, pos_pred=dec.pos)
    if dd is not None:
        frames.append((obs, obsb, dd))

obs0, obsb0, dd0 = frames[0]
print(f'{len(frames)} epochs collected')
print('epoch 0 common sats:', [int(s) for s in dd0.sat])
print('epoch 0 elevations [deg]:', np.round(np.rad2deg(dd0.el), 1))

59 epochs collected
epoch 0 common sats: [3, 4, 6, 9, 14, 17, 19, 22, 28, 35, 39, 40, 45, 47, 53, 58]
epoch 0 elevations [deg]: [40.8 35.7 40.9 33.  25.2 85.4 61.6 16.  32.1 32.8 17.9 48.6 60.9 41.4
 27.8 18.7]


### 3. Reference satellite per constellation (gauge)

Only between-satellite double differences are observable, so one single-difference
ambiguity per constellation is unobservable. We pick the highest-elevation
satellite as the reference and later pin its ambiguity (any value works -- the DDs
are gauge-independent).

In [5]:
el_cum = {}
for (_, _, dd) in frames:
    for k, s in enumerate(dd.sat):
        if dd.el[k] > 0:
            el_cum[int(s)] = el_cum.get(int(s), 0.0) + dd.el[k]
ref_of = {}
for s, e in el_cum.items():
    sys = sat2prn(s)[0]
    if sys in SYSS and (sys not in ref_of or e > el_cum[ref_of[sys]]):
        ref_of[sys] = s
print('reference satellite per constellation:',
      {int(k): int(v) for k, v in ref_of.items()})

reference satellite per constellation: {0: 17, 1: 45}


### 4. Build the float factor graph incrementally (ISAM2)

For every epoch and every frequency we add, per target satellite:
a `DoubleDifferencePseudorangeFactor` and a `DoubleDifferenceCarrierPhaseFactor`
(both form the rover-base double difference internally via Sagnac-corrected
`gnss::geodist`). The carrier factor ties the reference and target
**between-receiver SD ambiguities**; the reference ambiguity is gauge-pinned to its
carrier-minus-code value (non-zero so cssrlib's `ddidx` can pivot on it).

ISAM2 uses **QR** factorization, whose joint marginals are robust. This loop builds
only the **float** solution -- integer resolution comes afterwards in Section 5.

In [6]:
params = gtsam.ISAM2Params(); params.setFactorization('QR')
isam = gtsam.ISAM2(params)
seen_am, pinned = set(), set()
nfac = 0

for ei, (obs, obsb, dd) in enumerate(frames):
    graph = gtsam.NonlinearFactorGraph()
    val = gtsam.Values()
    if ei == 0:
        val.insert(X, gtsam.Point3(*dec.pos))
        graph.add(gtsam.PriorFactorPoint3(
            X, gtsam.Point3(*dec.pos), gtsam.noiseModel.Isotropic.Sigma(3, 30.0)))

    by_sys = {}
    for k, s in enumerate(dd.sat):
        by_sys.setdefault(sat2prn(int(s))[0], []).append(k)
    for sys, ks in by_sys.items():
        ref = ref_of.get(sys)
        ridx = next((k for k in ks if int(dd.sat[k]) == ref), None)
        if ridx is None:
            continue
        for f in range(nf):
            lam = obs.sig[sys][uTYP.L][f].wavelength()
            pr_rr, pr_br = obs.P[dd.iu[ridx], f], obsb.P[dd.ir[ridx], f]
            cp_rr = obs.L[dd.iu[ridx], f] * lam
            cp_br = obsb.L[dd.ir[ridx], f] * lam
            if 0.0 in (pr_rr, pr_br, cp_rr, cp_br):
                continue
            rs_ref, rsb_ref = dd.rs[dd.iu[ridx], :3], dd.rsb[dd.ir[ridx], :3]
            sd_ref = ((cp_rr - cp_br) - (pr_rr - pr_br)) / lam   # gauge value
            if AM(ref, f) not in seen_am:
                val.insert(AM(ref, f), float(sd_ref)); seen_am.add(AM(ref, f))
            if (ref, f) not in pinned:
                graph.addPriorDouble(AM(ref, f), sd_ref,
                                     gtsam.noiseModel.Isotropic.Sigma(1, 0.5))
                pinned.add((ref, f))
            for k in ks:
                js = int(dd.sat[k])
                if k == ridx:
                    continue
                pr_tr, pr_tb = obs.P[dd.iu[k], f], obsb.P[dd.ir[k], f]
                cp_tr = obs.L[dd.iu[k], f] * lam
                cp_tb = obsb.L[dd.ir[k], f] * lam
                if 0.0 in (pr_tr, pr_tb, cp_tr, cp_tb):
                    continue
                rs_j, rsb_j = dd.rs[dd.iu[k], :3], dd.rsb[dd.ir[k], :3]
                w = 1.0 / max(np.sin(min(dd.el[k], dd.el[ridx])), 0.1)
                graph.add(gtsam.DoubleDifferencePseudorangeFactor(
                    X, pr_rr, pr_br, pr_tr, pr_tb,
                    gtsam.Point3(*rs_ref), gtsam.Point3(*rs_j),
                    gtsam.Point3(*rsb_ref), gtsam.Point3(*rsb_j),
                    gtsam.Point3(*rb), gtsam.noiseModel.Isotropic.Sigma(1, 0.3 * w)))
                sd_tgt = ((cp_tr - cp_tb) - (pr_tr - pr_tb)) / lam
                if AM(js, f) not in seen_am:
                    val.insert(AM(js, f), float(sd_tgt)); seen_am.add(AM(js, f))
                graph.add(gtsam.DoubleDifferenceCarrierPhaseFactor(
                    X, AM(ref, f), AM(js, f), cp_rr, cp_br, cp_tr, cp_tb,
                    gtsam.Point3(*rs_ref), gtsam.Point3(*rs_j),
                    gtsam.Point3(*rsb_ref), gtsam.Point3(*rsb_j),
                    gtsam.Point3(*rb), lam,
                    gtsam.noiseModel.Isotropic.Sigma(1, 0.01 * w)))
    nfac += graph.size()
    isam.update(graph, val)

    res = isam.calculateEstimate()
    xh = np.array(res.atPoint3(X))
    enu = gn.ecef2enu(pos_ref, xh - xyz_ref)
    if ei % 10 == 0 or ei == len(frames) - 1:
        print(f'ep{ei:3d}  {"float":<5}  2D={np.hypot(enu[0], enu[1]):6.3f}  '
              f'3D={np.linalg.norm(xh - xyz_ref):6.3f} m')
print(f'\ngraph: {nfac} factors, {len(seen_am)} ambiguities')
res = isam.calculateEstimate()

ep  0  float  2D= 0.088  3D= 0.089 m
ep 10  float  2D= 0.131  3D= 0.258 m
ep 20  float  2D= 0.161  3D= 0.251 m
ep 30  float  2D= 0.161  3D= 0.252 m
ep 40  float  2D= 0.167  3D= 0.289 m


ep 50  float  2D= 0.159  3D= 0.289 m


ep 58  float  2D= 0.155  3D= 0.262 m

graph: 3393 factors, 34 ambiguities


### 5. Integer ambiguity resolution (explicit)

Now we resolve the integers on the final float estimate. The bridge from GTSAM to
cssrlib's LAMBDA (`resamb_lambda`) is unfolded here step by step instead of hidden
in a helper:

1. Write the float SD ambiguities into the cssrlib `nav` state (`nav.x`).
2. Write the **position covariance** (`isam.marginalCovariance(X)`) into `nav.P`.
3. Write the **ambiguity covariance**. The full position+ambiguity joint marginal is
   numerically ill-conditioned (Point3 mixed with many correlated cycle-scale
   ambiguities -> NaN), so it is assembled from the *ambiguity-only* joint (stable)
   plus *pairwise* (position, ambiguity) cross terms.
4. Guard any non-finite entry, then call `resamb_lambda` (LAMBDA + `ddidx`
   single-difference mapping + ratio test). Only the AR *algorithm* is used -- not
   cssrlib's EKF.

In [7]:
nav = rtk.nav

# (1) float ambiguities -> nav.x ; reset the rest of the state
nav.x[nav.na:] = 0.0
nav.P[:, :] = 0.0
nav.vsat[:, :] = 0
nav.x[0:3] = np.array(res.atPoint3(X))

# the SD ambiguities present in the graph (use the last epoch's sat/el)
obs, obsb, dd = frames[-1]
amb = [(int(s), f) for s in dd.sat for f in range(nf)
       if sat2prn(int(s))[0] in SYSS and AM(int(s), f) in seen_am
       and res.exists(AM(int(s), f))]
el_now = {int(s): dd.el[i] for i, s in enumerate(dd.sat)}
for (s, f) in amb:
    j = rtk.IB(s, f, nav.na)
    nav.x[j] = res.atDouble(AM(s, f))
    nav.vsat[s - 1, f] = 1
    nav.el[s - 1] = el_now[s]
print(f'{len(amb)} float ambiguities written into nav.x')

# (2) position covariance (ISAM2 QR -> robust marginal)
P_pos = isam.marginalCovariance(X)
nav.P[0:3, 0:3] = P_pos
print(f'position 1-sigma = {np.sqrt(np.trace(P_pos)):.3f} m')

# (3a) ambiguity-only joint marginal (stable; full X+amb joint would be NaN)
kv = gtsam.KeyVector([AM(s, f) for (s, f) in amb])
jm = isam.jointMarginalCovariance(kv)
for (s, f) in amb:
    j = rtk.IB(s, f, nav.na)
    nav.P[j, j] = jm.at(AM(s, f), AM(s, f))[0, 0]
    # (3b) pairwise (position, ambiguity) cross-covariance
    pxn = isam.jointMarginalCovariance(
        gtsam.KeyVector([X, AM(s, f)])).at(X, AM(s, f))[:, 0]
    nav.P[0:3, j] = pxn; nav.P[j, 0:3] = pxn
# (3c) ambiguity-ambiguity cross terms from the ambiguity-only joint
for a in range(len(amb)):
    s1, f1 = amb[a]; j1 = rtk.IB(s1, f1, nav.na)
    for b in range(a + 1, len(amb)):
        s2, f2 = amb[b]; j2 = rtk.IB(s2, f2, nav.na)
        c = jm.at(AM(s1, f1), AM(s2, f2))[0, 0]
        nav.P[j1, j2] = c; nav.P[j2, j1] = c

# (4) guard non-finite entries, then run LAMBDA
bad = ~np.isfinite(nav.P)
if bad.any():
    nav.P[bad] = 0.0
    d = np.where(np.diag(bad))[0]; nav.P[d, d] = 1e10
nav.elmaskar = np.deg2rad(15.0)
sat_ar = np.array(sorted({s for (s, f) in amb}))
nb, _ = rtk.resamb_lambda(sat_ar, nav.parmode, nav.par_P0)
fixed_xyz = np.array(nav.xa[0:3]) if nb > 0 else None

# show float vs fixed as aligned rows (same layout as the Section 4 table)
xf = np.array(res.atPoint3(X))
enu_f = gn.ecef2enu(pos_ref, xf - xyz_ref)
print(f'ep{len(frames) - 1:3d}  {"float":<5}  2D={np.hypot(enu_f[0], enu_f[1]):6.3f}  '
      f'3D={np.linalg.norm(xf - xyz_ref):6.3f} m')
if fixed_xyz is not None:
    enu_x = gn.ecef2enu(pos_ref, fixed_xyz - xyz_ref)
    print(f'{"fix":<5}  {"FIX":<5}  2D={np.hypot(enu_x[0], enu_x[1]):6.3f}  '
          f'3D={np.linalg.norm(fixed_xyz - xyz_ref):6.3f} m  ({nb} amb)')

32 float ambiguities written into nav.x
position 1-sigma = 0.052 m
ep 58  float  2D= 0.155  3D= 0.262 m
fix    FIX    2D= 0.007  3D= 0.015 m  (28 amb)


### 6. Results

With ~23 satellites on two frequencies and a short baseline, RTK fixes
**instantaneously** and the fixed solution agrees with the surveyed marker at the
**mm-cm** level (the float solution is already dm-level; the integer fix tightens it).

In [8]:
xf = np.array(res.atPoint3(X))
enu_f = gn.ecef2enu(pos_ref, xf - xyz_ref)
print(f'final  {"float":<5}  2D={np.hypot(enu_f[0], enu_f[1]):6.3f}  '
      f'3D={np.linalg.norm(xf - xyz_ref):6.3f} m')
if fixed_xyz is not None:
    enu_x = gn.ecef2enu(pos_ref, fixed_xyz - xyz_ref)
    print(f'final  {"FIX":<5}  2D={np.hypot(enu_x[0], enu_x[1]):6.3f}  '
          f'3D={np.linalg.norm(fixed_xyz - xyz_ref):6.3f} m  ({nb} SD ambiguities)')

final  float  2D= 0.155  3D= 0.262 m
final  FIX    2D= 0.007  3D= 0.015 m  (28 SD ambiguities)


## Part 2 - PPP-RTK (undifferenced, single receiver + CLAS)

Now drop the base station. Without differencing, the receiver clock, tropospheric zenith wet delay and slant ionosphere no longer cancel, so they become state variables, corrected by QZSS CLAS (broadcast on L6). This is where the **undifferenced** factors are used.

In [9]:
import os
from copy import deepcopy
from binascii import unhexlify
import numpy as np

from cssrlib.cssrlib import cssr
from cssrlib.gnss import (ecef2pos, ecef2enu, Nav, time2gpst, time2doy, rSigRnx,
                          epoch2time, sat2prn, uGNSS)
from cssrlib.gnss import geodist as cssr_geodist
from cssrlib.peph import atxdec, searchpcv
from cssrlib.ppprtk import ppprtkpos
from cssrlib.rinex import rnxdec
import gtsam
from gtsam import symbol

SYSS = (uGNSS.GPS, uGNSS.GAL, uGNSS.QZS)
X = symbol('x', 0)
def CK(ei, si): return symbol('c', ei * 4 + si)   # per-epoch, per-system clock
def ZT(ei): return symbol('z', ei)                # per-epoch ZTD (random walk)
def IO(s): return symbol('i', int(s))             # per-sat slant iono
def AM(s, f): return symbol('n', int(s) * 4 + f)  # per-sat/freq ambiguity

### 1. Load RINEX, ANTEX and the CLAS grid

`ppprtkpos` is the cssrlib PPP-RTK front-end. `xyz_ref` is the surveyed marker
(accuracy scoring only).

In [10]:
# DATADIR is set in the setup cell above
ep = [2025, 8, 21, 7, 0, 0]
xyz_ref = np.array([-3962108.7007, 3381309.5532, 3668678.6648])
pos_ref = ecef2pos(xyz_ref)
NEP = int(os.environ.get('NEP', '120'))
time = epoch2time(ep); doy = int(time2doy(time)); let = chr(ord('a') + ep[3])
bdir = f'{DATADIR}/doy{ep[0]:04d}-{doy:03d}/'

sigs = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("EC1C"), rSigRnx("EC5Q"),
        rSigRnx("JC1C"), rSigRnx("JC2L"),
        rSigRnx("GL1C"), rSigRnx("GL2W"), rSigRnx("EL1C"), rSigRnx("EL5Q"),
        rSigRnx("JL1C"), rSigRnx("JL2L"),
        rSigRnx("GS1C"), rSigRnx("GS2W"), rSigRnx("ES1C"), rSigRnx("ES5Q"),
        rSigRnx("JS1C"), rSigRnx("JS2L")]
nav = Nav(); nav = rnxdec().decode_nav(bdir + f'{doy:03d}{let}_rnx.nav', nav)
atx = atxdec(); atx.readpcv(f'{DATADIR}/antex/igs20.atx')
rnx = rnxdec(); rnx.setSignals(sigs)
cs = cssr(); cs.monlevel = 0; cs.week = time2gpst(time)[0]
cs.read_griddef(f'{DATADIR}/clas_grid.def')
assert rnx.decode_obsh(bdir + f'{doy:03d}{let}_rnx.obs') >= 0
rnx.autoSubstituteSignals()
ppp = ppprtkpos(nav, rnx.pos)
nav.rcv_ant = searchpcv(atx.pcvr, rnx.ant, rnx.ts); nav.sat_ant = atx.pcvs
cs.find_grid_index(ecef2pos(rnx.pos)); nf = nav.nf
print('frequencies:', nf, ' constellations:', [int(s) for s in SYSS])

frequencies: 2  constellations: [0, 1, 2]


### 2. Decode CLAS (L6) and build the front-end measurements

For each epoch the L6 stream is decoded; once a full CLAS set is available,
`prepare_ppp_measurements` returns SSR-corrected undifferenced residuals (`y`),
satellite states (`rs`), tropo mapping (`mapfw`), iono coefficients (`mu`),
wavelengths (`lam`) and the CLAS atmosphere a-priori sigmas (`iono_sig`/`ztd_sig`).

In [11]:
v = np.genfromtxt(bdir + f'{doy:03d}{let}_qzsl6.txt',
                  dtype=[('wn', 'int'), ('tow', 'int'), ('prn', 'int'),
                         ('type', 'int'), ('len', 'int'), ('nav', 'S500')])
frames = []
obs = rnx.decode_obs()
while time > obs.t and obs.t.time != 0:
    obs = rnx.decode_obs()
for k in range(NEP):
    week, tow = time2gpst(obs.t)
    vi = v[(v['tow'] == tow) & (v['type'] == 0) & (v['prn'] == 199)]
    if len(vi) > 0:
        cs.decode_l6msg(unhexlify(vi['nav'][0]), 0)
        if cs.fcnt == 5:
            cs.decode_cssr(bytes(cs.buff), 0)
    if k == 0:
        nav.t = deepcopy(obs.t); t0 = deepcopy(obs.t)
        t0.time = t0.time // 30 * 30; cs.time = obs.t; nav.time_p = t0
    if cs.chk_stat():
        ppm = ppp.prepare_ppp_measurements(obs, cs=cs, pos_pred=rnx.pos)
        if ppm is not None and k >= 20:
            frames.append(ppm)
    obs = rnx.decode_obs()
    if obs.t.time == 0:
        break
print(f'{len(frames)} epochs collected via prepare_ppp_measurements')

100 epochs collected via prepare_ppp_measurements


### 3. State model and the random-walk tropo factor

PPP cannot difference errors away, so they become states: a per-epoch per-system
**clock** `CK(ei, sys)`, a per-epoch **ZTD** `ZT(ei)` linked across epochs by a
random-walk factor, a per-satellite slant **iono** `IO(s)` (tight CLAS STEC prior),
and per-satellite/frequency **ambiguities** `AM(s, f)`. ISAM2 uses **QR**.

In [12]:
params = gtsam.ISAM2Params(); params.setFactorization('QR')
isam = gtsam.ISAM2(params)
x0 = xyz_ref + np.array([5.0, -4.0, 3.0])      # deliberately ~7 m off
ztd_sigs = [fr.ztd_sig for fr in frames if np.isfinite(fr.ztd_sig)]
ztd_sig = float(np.median(ztd_sigs)) if ztd_sigs else 0.1


def ztd_rw():
    def err(this, values, jac):
        a = values.atDouble(this.keys()[0]); b = values.atDouble(this.keys()[1])
        if jac is not None:
            jac[0] = np.array([[1.0]]); jac[1] = np.array([[-1.0]])
        return np.array([a - b])
    return err

### 4. Build the float graph incrementally (ISAM2)

Per epoch: add the ZTD prior (epoch 0) or random-walk link; then for each
satellite/frequency add `UndifferencedPseudorangeFactor` and
`UndifferencedCarrierPhaseFactor` wired to position, the system clock, ZTD, the
slant iono and the ambiguity. We require both frequencies so the per-satellite
iono is observable. This loop builds only the **float** solution; the position
converges over the first epochs (PPP has no base to difference against).

In [13]:
seen_io, seen_am, seen_ck = set(), set(), set()
nfac = 0
for ei, fr in enumerate(frames):
    graph = gtsam.NonlinearFactorGraph(); val = gtsam.Values()
    if ei == 0:
        val.insert(X, gtsam.Point3(*x0))
        graph.add(gtsam.PriorFactorPoint3(X, gtsam.Point3(*x0),
                  gtsam.noiseModel.Isotropic.Sigma(3, 30.0)))
    rr = fr.pos_pred
    val.insert(ZT(ei), 0.0)
    if ei == 0:
        graph.addPriorDouble(ZT(0), 0.0,
                             gtsam.noiseModel.Isotropic.Sigma(1, ztd_sig))
    else:
        graph.add(gtsam.CustomFactor(
            gtsam.noiseModel.Isotropic.Sigma(1, 0.003),
            gtsam.KeyVector([ZT(ei), ZT(ei - 1)]), ztd_rw()))
    for i, s in enumerate(fr.sat):
        s = int(s); sys = sat2prn(s)[0]
        if sys not in SYSS or fr.el[i] <= 0:
            continue
        if not np.all(np.isfinite(fr.rs[i])) or np.linalg.norm(fr.rs[i]) < 1e6:
            continue
        if not (fr.y[i, 0] != 0 and fr.y[i, 1] != 0
                and fr.y[i, nf] != 0 and fr.y[i, nf + 1] != 0):
            continue
        geom, _ = cssr_geodist(fr.rs[i], rr)
        s_el = 1.0 / max(np.sin(fr.el[i]), 0.1)
        ck = CK(ei, SYSS.index(sys))
        for f in range(nf):
            lam, mu = fr.lam[i, f], fr.mu[i, f]
            if lam <= 0 or mu <= 0 or fr.y[i, f] == 0 or fr.y[i, nf + f] == 0:
                continue
            m_phase = fr.y[i, f] + geom
            m_code = fr.y[i, nf + f] + geom
            if ck not in seen_ck:
                seen_ck.add(ck); val.insert(ck, float(m_code - geom))
                graph.addPriorDouble(ck, 0.0,
                                     gtsam.noiseModel.Isotropic.Sigma(1, 1e5))
            if IO(s) not in seen_io:
                seen_io.add(IO(s)); val.insert(IO(s), 0.0)
                sig_i = min(fr.iono_sig[i] if np.isfinite(fr.iono_sig[i])
                            else 0.05, 0.01)
                graph.addPriorDouble(IO(s), 0.0,
                                     gtsam.noiseModel.Isotropic.Sigma(1, sig_i))
            graph.add(gtsam.UndifferencedPseudorangeFactor(
                X, ck, ZT(ei), IO(s), m_code, gtsam.Point3(*fr.rs[i]),
                fr.mapfw[i], mu, 0.0,
                gtsam.noiseModel.Isotropic.Sigma(1, 0.6 * s_el)))
            ak = AM(s, f)
            if ak not in seen_am:
                seen_am.add(ak)
                val.insert(ak, float((m_phase - geom - val.atDouble(ck)) / lam))
                graph.addPriorDouble(ak, val.atDouble(ak),
                                     gtsam.noiseModel.Isotropic.Sigma(1, 5.0))
            graph.add(gtsam.UndifferencedCarrierPhaseFactor(
                X, ck, ZT(ei), IO(s), ak, m_phase, gtsam.Point3(*fr.rs[i]),
                fr.mapfw[i], mu, lam, 0.0,
                gtsam.noiseModel.Isotropic.Sigma(1, 0.006 * s_el)))
    nfac += graph.size()
    isam.update(graph, val)

    res = isam.calculateEstimate()
    xh = np.array(res.atPoint3(X))
    enu = ecef2enu(pos_ref, xh - xyz_ref)
    if ei % 15 == 0 or ei == len(frames) - 1:
        print(f'ep{ei:3d}  {"float":<5}  2D={np.hypot(enu[0], enu[1]):6.3f}  '
              f'3D={np.linalg.norm(xh - xyz_ref):6.3f} m')
print(f'\ngraph: {nfac} factors, {len(seen_am)} ambiguities')
res = isam.calculateEstimate()

ep  0  float  2D= 0.281  3D= 0.452 m
ep 15  float  2D= 0.070  3D= 0.459 m
ep 30  float  2D= 0.182  3D= 1.254 m
ep 45  float  2D= 0.041  3D= 0.868 m


ep 60  float  2D= 0.248  3D= 1.093 m
ep 75  float  2D= 0.157  3D= 0.599 m


ep 90  float  2D= 0.174  3D= 0.604 m


ep 99  float  2D= 0.208  3D= 0.220 m

graph: 5571 factors, 28 ambiguities


### 5. Integer ambiguity resolution (explicit), with convergence gate

Same bridge as RTK, unfolded explicitly: write the float ambiguities and their
covariance into the cssrlib `nav` state, then call `resamb_lambda`. The full
position+ambiguity joint marginal is ill-conditioned, so the covariance is
assembled from the *ambiguity-only* joint (stable) plus *pairwise* (position,
ambiguity) cross terms.

Unlike RTK (where pseudorange pins the position directly), in PPP-RTK the position
is co-estimated with clock/ZTD/iono and only becomes observable after a few epochs,
so we first check a **convergence gate**: AR is attempted only once the position
1-sigma is below `conv_sigma = 1.0` m. Here we run it on the fully converged final
estimate.

In [14]:
nav = ppp.nav
conv_sigma = 1.0

# (1) float ambiguities -> nav.x ; reset the rest of the state
nav.x[nav.na:] = 0.0
nav.P[:, :] = 0.0
nav.vsat[:, :] = 0
nav.x[0:3] = np.array(res.atPoint3(X))

# convergence gate: position 1-sigma from ISAM2 (QR -> robust marginal)
P_pos = isam.marginalCovariance(X)
pos_sigma = np.sqrt(np.trace(P_pos))
print(f'position 1-sigma = {pos_sigma:.3f} m '
      f'({"converged" if pos_sigma <= conv_sigma else "NOT converged"})')

fr = frames[-1]
amb = [(int(s), f) for s in fr.sat for f in range(nf)
       if sat2prn(int(s))[0] in SYSS and AM(int(s), f) in seen_am
       and res.exists(AM(int(s), f))]
el_now = {int(s): fr.el[i] for i, s in enumerate(fr.sat)}

if pos_sigma <= conv_sigma and len(amb) >= 4:
    for (s, f) in amb:
        j = ppp.IB(s, f, nav.na)
        nav.x[j] = res.atDouble(AM(s, f))
        nav.vsat[s - 1, f] = 1
        if s in el_now:
            nav.el[s - 1] = el_now[s]
    nav.P[0:3, 0:3] = P_pos

    # (3a) ambiguity-only joint marginal (stable; full X+amb joint is NaN)
    kv = gtsam.KeyVector([AM(s, f) for (s, f) in amb])
    jm = isam.jointMarginalCovariance(kv)
    for (s, f) in amb:
        j = ppp.IB(s, f, nav.na)
        nav.P[j, j] = jm.at(AM(s, f), AM(s, f))[0, 0]
        # (3b) pairwise (position, ambiguity) cross-covariance
        pxn = isam.jointMarginalCovariance(
            gtsam.KeyVector([X, AM(s, f)])).at(X, AM(s, f))[:, 0]
        nav.P[0:3, j] = pxn; nav.P[j, 0:3] = pxn
    # (3c) ambiguity-ambiguity cross terms from the ambiguity-only joint
    for a in range(len(amb)):
        s1, f1 = amb[a]; j1 = ppp.IB(s1, f1, nav.na)
        for b in range(a + 1, len(amb)):
            s2, f2 = amb[b]; j2 = ppp.IB(s2, f2, nav.na)
            cc = jm.at(AM(s1, f1), AM(s2, f2))[0, 0]
            nav.P[j1, j2] = cc; nav.P[j2, j1] = cc

    # (4) guard non-finite entries, then run LAMBDA
    bad = ~np.isfinite(nav.P)
    if bad.any():
        nav.P[bad] = 0.0
        d = np.where(np.diag(bad))[0]; nav.P[d, d] = 1e10
    nav.elmaskar = np.deg2rad(15.0)
    sat_ar = np.array(sorted({s for (s, f) in amb}))
    nb, _ = ppp.resamb_lambda(sat_ar, nav.parmode, nav.par_P0)
    fixed_xyz = np.array(nav.xa[0:3]) if nb > 0 else None
else:
    nb, fixed_xyz = 0, None

# show float vs fixed as aligned rows (same layout as the Section 4 table)
xf = np.array(res.atPoint3(X))
enu_f = ecef2enu(pos_ref, xf - xyz_ref)
print(f'ep{len(frames) - 1:3d}  {"float":<5}  2D={np.hypot(enu_f[0], enu_f[1]):6.3f}  '
      f'3D={np.linalg.norm(xf - xyz_ref):6.3f} m')
if fixed_xyz is not None:
    enu_x = ecef2enu(pos_ref, fixed_xyz - xyz_ref)
    print(f'{"fix":<5}  {"FIX":<5}  2D={np.hypot(enu_x[0], enu_x[1]):6.3f}  '
          f'3D={np.linalg.norm(fixed_xyz - xyz_ref):6.3f} m  ({nb} amb)')
else:
    print('AR skipped (not converged or too few ambiguities)')

position 1-sigma = 0.103 m (converged)
ep 99  float  2D= 0.208  3D= 0.220 m
fix    FIX    2D= 0.022  3D= 0.074 m  (22 amb)


### 6. Results

CLAS PPP-RTK converges and fixes to the **cm** level against the surveyed marker --
with a single receiver, no base station. (In the per-epoch streaming version the
convergence gate lets it fix from ~epoch 1.)

In [15]:
xf = np.array(res.atPoint3(X))
enu_f = ecef2enu(pos_ref, xf - xyz_ref)
print(f'final  {"float":<5}  2D={np.hypot(enu_f[0], enu_f[1]):6.3f}  '
      f'3D={np.linalg.norm(xf - xyz_ref):6.3f} m')
if fixed_xyz is not None:
    enu_x = ecef2enu(pos_ref, fixed_xyz - xyz_ref)
    print(f'final  {"FIX":<5}  2D={np.hypot(enu_x[0], enu_x[1]):6.3f}  '
          f'3D={np.linalg.norm(fixed_xyz - xyz_ref):6.3f} m  ({nb} SD ambiguities)')

final  float  2D= 0.208  3D= 0.220 m
final  FIX    2D= 0.022  3D= 0.074 m  (22 SD ambiguities)


## Sources
- [PseudorangeFactor.h](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/PseudorangeFactor.h) / [.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/PseudorangeFactor.cpp)
- [CarrierPhaseFactor.h](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/CarrierPhaseFactor.h) / [.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/CarrierPhaseFactor.cpp)
- [PseudorangeFactor.ipynb](../../../gtsam/navigation/doc/PseudorangeFactor.ipynb)
- [DifferentialPseudorangeExample.ipynb](DifferentialPseudorangeExample.ipynb)
- GNSS front-end: [cssrlib](https://github.com/inuex35/cssrlib-numba), data: [cssrlib-data](https://github.com/hirokawa/cssrlib-data)